1. Setup & import

In [ ]:
import sys  #Tương tác với trình biên dịch python
sys.path.append("..")  # thêm đường dẫn python có thể để import được từ src/ (Thư mục khác cha với load_imdb_data) 
# ".." tức là lùi lên một cấp -> cấp cha

from src.load_data import load_imdb_data
from src.preprocessing import preprocess_pipeline, load_teencode_dict
import pandas as pd
import time
import os

2. Load data + teencode_dict

In [3]:
df = load_imdb_data(r"D:\Coding\Independent Projects\Sentiment_Analysis_API\data\raw\IMDB Dataset.csv")
teencode_map =  load_teencode_dict(r"D:\Coding\Independent Projects\Sentiment_Analysis_API\teencode_dict.json")

print(f"Tổng số dòng: {len(df)}")
print(f"Số cặp teencode đã load: {len(teencode_map)}")

# Kiểm tra nhanh dữ liệu trước khi xử lý toàn bộ (tránh chạy full rồi mới phát hiện lỗi)
assert "review" in df.columns
assert df["review"].isna().sum() == 0, "Có giá trị null trong cột review, cần xử lý trước"

[INFO] Phát hiện 418 review trùng lặp - sẽ loại bỏ.
Tổng số dòng: 49582
Số cặp teencode đã load: 30


3. Test trên sample nhỏ

In [4]:
# Test trên 10 dòng đầu trước khi chạy full 50k — bắt lỗi sớm
sample = df["review"].head(10)
for text in sample:
    result = preprocess_pipeline(text, teencode_map)
    print(text[:80], "→", result[:80])
    print("---")

One of the other reviewers has mentioned that after watching just 1 Oz episode y → one of the other reviewers has mentioned that after watching just 1 oz episode y
---
A wonderful little production. <br /><br />The filming technique is very unassum → a wonderful little production. the filming technique is very unassuming- very ol
---
I thought this was a wonderful way to spend time on a too hot summer weekend, si → i thought this was a wonderful way to spend time on a too hot summer weekend, si
---
Basically there's a family where a little boy (Jake) thinks there's a zombie in  → basically there's a family where a little boy (jake) thinks there's a zombie in 
---
Petter Mattei's "Love in the Time of Money" is a visually stunning film to watch → petter mattei's "love in the time of money" is a visually stunning film to watch
---
Probably my all-time favorite movie, a story of selflessness, sacrifice and dedi → probably my all-time favorite movie, a story of selflessness, sacrifice and d

Test sample trả kết quả OK

4. Áp dụng lên toàn bộ dataset, có đo thời gian

In [5]:
start = time.time()

df["review_clean"] = df["review"].apply(
    lambda text: preprocess_pipeline(text, teencode_map)
)

elapsed = time.time() - start
print(f"Xử lý {len(df):,} dòng mất {elapsed:.1f} giây "
      f"({elapsed/len(df)*1000:.2f} ms/dòng)")

Xử lý 49,582 dòng mất 72.1 giây (1.45 ms/dòng)


5. Kiểm tra kết quả trước khi lưu

In [6]:
# Không được có null sau khi xử lý
assert df["review_clean"].isna().sum() == 0, "Có giá trị null sau preprocessing"

# Không được có chuỗi rỗng bất thường (trừ khi input gốc cũng rỗng)
empty_after = (df["review_clean"].str.strip() == "").sum()
empty_before = (df["review"].str.strip() == "").sum()
print(f"Chuỗi rỗng sau xử lý: {empty_after} (trước xử lý: {empty_before})")

# Xem nhanh vài dòng để kiểm tra bằng mắt
df[["review", "review_clean"]].sample(3, random_state=42)

Chuỗi rỗng sau xử lý: 0 (trước xử lý: 0)


,review,review_clean
29035,"""Soul Plane"" is a horrible attempt at comedy t...","""soul plane"" is a horrible attempt at comedy t..."
43282,Guest from the Future tells a fascinating stor...,guest from the future tells a fascinating stor...
38461,"""National Treasure"" (2004) is a thoroughly mis...","""national treasure"" (2004) is a thoroughly mis..."


6. Lưu ra file, giữ nguyên cả cột raw

In [7]:
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/imdb_processed.csv"
df.to_csv(output_path, index=False)

print(f"Đã lưu {len(df):,} dòng vào {output_path}")
print(f"Kích thước file: {os.path.getsize(output_path) / 1_048_576:.1f} MB")
print(f"Các cột: {list(df.columns)}")

Đã lưu 49,582 dòng vào ../data/processed/imdb_processed.csv
Kích thước file: 123.9 MB
Các cột: ['review', 'sentiment', 'review_clean']


7. EDA đối chiếu với eda_raw

In [13]:
from collections import Counter

def get_top_words(texts, n=20):
    words = " ".join(texts).lower().split()
    return Counter(words).most_common(n)
pos_words_clean = get_top_words(df[df["sentiment"] == "positive"]["review_clean"])
neg_words_clean = get_top_words(df[df["sentiment"] == "negative"]["review_clean"])

print("Top 20 từ - positive:", pos_words_clean)
print("Top 20 từ - negative:", neg_words_clean)

pos_words_raw = get_top_words(df[df["sentiment"] == "positive"]["review"])
neg_words_raw = get_top_words(df[df["sentiment"] == "negative"]["review"])

def print_comparison(label, raw_words, clean_words):
    print(f"=== {label} — TRƯỚC preprocessing ===")
    for word, count in raw_words:
        print(f"  {word:15s} {count}")
    print(f"\n=== {label} — SAU preprocessing ===")
    for word, count in clean_words:
        print(f"  {word:15s} {count}")
    print()

print_comparison("POSITIVE", pos_words_raw, pos_words_clean)
print_comparison("NEGATIVE", neg_words_raw, neg_words_clean)

# 1. "br" đã biến mất khỏi top từ chưa
raw_words_set = {w for w, _ in pos_words_raw + neg_words_raw}
clean_words_set = {w for w, _ in pos_words_clean + neg_words_clean}

print("'br' còn trong top từ RAW:", "br" in raw_words_set)
print("'br' còn trong top từ CLEAN:", "br" in clean_words_set)
assert "br" not in clean_words_set, "clean_html() chưa hoạt động đúng — 'br' vẫn còn trong top từ"

# 2. Không còn ký tự emoji thô nào trong review_clean
import emoji as emoji_lib

n_emoji_remaining = df["review_clean"].apply(
    lambda t: bool(emoji_lib.emoji_count(t))
).sum()

print(f"Số dòng còn sót emoji thô sau xử lý: {n_emoji_remaining}")
assert n_emoji_remaining == 0, "translate_emoji() chưa xử lý hết emoji"

# 3. Teencode phổ biến không còn dạng viết tắt (kiểm tra 1 vài ví dụ cụ thể)
# Ví dụ: "lol" → "laugh out loud" (3 từ tách rời trong get_top_words, đây là expected)
sample_slang = list(teencode_map.keys())[:5]  # lấy vài slang đầu để kiểm tra mẫu

for slang in sample_slang:
    n_before = df["review"].str.lower().str.contains(rf"\b{slang}\b", regex=True).sum()
    n_after = df["review_clean"].str.contains(rf"\b{slang}\b", regex=True).sum()
    print(f"'{slang}': xuất hiện {n_before} lần (raw) → {n_after} lần (clean)")

Top 20 từ - positive: [('the', 332539), ('and', 171343), ('a', 161275), ('of', 150507), ('to', 129407), ('is', 108813), ('in', 96433), ('i', 69993), ('this', 66397), ('it', 65599), ('that', 63845), ('as', 49917), ('with', 44481), ('for', 43023), ('was', 42190), ('but', 38613), ('his', 33302), ('on', 31525), ('film', 29420), ('are', 28528)]
Top 20 từ - negative: [('the', 316453), ('a', 154621), ('and', 141461), ('of', 134617), ('to', 133811), ('is', 95061), ('in', 84207), ('i', 77859), ('this', 75812), ('that', 65751), ('it', 65195), ('was', 50416), ('for', 41574), ('but', 40176), ('with', 39840), ('as', 39123), ('movie', 34667), ('on', 31312), ('not', 30001), ('have', 29953)]
=== POSITIVE — TRƯỚC preprocessing ===
  the             325111
  and             170782
  a               160440
  of              150236
  to              129177
  is              108654
  in              95343
  i               66516
  it              64530
  this            63902
  that            63643
  as  

Đo mức độ ảnh hưởng thực tế

1. % review chứa ít nhất 1 teencode

In [14]:
import re

def contains_teencode_regex(text, teencode_map):
    text_lower = text.lower()
    return any(
        re.search(rf"\b{re.escape(slang)}\b", text_lower)
        for slang in teencode_map.keys()
    )

n_teencode_v2 = df["review"].apply(lambda t: contains_teencode_regex(t, teencode_map)).sum()
pct_teencode_v2 = n_teencode_v2 / len(df) * 100

print(f"Số review chứa teencode (regex word-boundary): {n_teencode_v2:,} ({pct_teencode_v2:.2f}%)")

Số review chứa teencode (regex word-boundary): 1,086 (2.19%)


2. % review chứa emoji

In [15]:
n_emoji = df["review"].apply(lambda t: bool(emoji_lib.emoji_count(t))).sum()
pct_emoji = n_emoji / len(df) * 100

print(f"Số review chứa emoji: {n_emoji:,} / {len(df):,} ({pct_emoji:.2f}%)")

Số review chứa emoji: 5 / 49,582 (0.01%)


Tổng hợp số liệu vào 1 bảng, dễ copy vào README

In [16]:
summary = pd.DataFrame({
    "Metric": [
        "Tổng số review",
        "% review chứa teencode",
        "% review chứa emoji",
        "% review chứa HTML tag (từ Ngày 2)",
    ],
    "Value": [
        f"{len(df):,}",
        f"{pct_teencode_v2:.2f}%",
        f"{pct_emoji:.2f}%",
        "~91%",  # điền lại số chính xác đã ghi ở day02_eda_raw_completed.md
    ]
})
print(summary.to_string(index=False))

summary.to_csv("../reports/preprocessing_impact_summary.csv", index=False)

                            Metric  Value
                    Tổng số review 49,582
            % review chứa teencode  2.19%
               % review chứa emoji  0.01%
% review chứa HTML tag (từ Ngày 2)   ~91%
